# 📓 Notebook 02 : Préparation des Données, Masques de Perlin & Contrôle Visuel
**Projet :** LAAFI_AI IVA Engine (Version 2.0)
**Objectif :** Pré-générer les masques de bruit biologiques hors-ligne, instancier la classe `IVADataset` et réaliser un contrôle visuel sur 5 échantillons augmentés.

In [ ]:
# 1. Importation des Modules du Package Modulaire
import sys
import os
import matplotlib.pyplot as plt
import torch

# Ajout du dossier racine au PATH
sys.path.append(os.path.abspath('..'))

from src.utils.seed import seed_everything
from src.data.generate_perlin_masks import generate_perlin_masks
from src.data.cluster_patients import generate_patient_clusters_and_splits
from src.data.dataset import IVADataset

seed_everything(42)

In [ ]:
# 2. Pré-génération des Masques de Bruit de Perlin (1 000 Masques .npy)
generate_perlin_masks(output_dir="../data/synthetic_masks", num_masks=1000)

In [ ]:
# 3. Génération des Splits Patients Étanches (GroupKFold)
generate_patient_clusters_and_splits(data_raw_dir="../data/raw", output_dir="../data/processed")

In [ ]:
# 4. Instanciation du PyTorch Dataset Class avec Augmentations & Bruit
train_dataset = IVADataset(
    csv_file="../data/processed/train.csv",
    is_train=True,
    masks_dir="../data/synthetic_masks",
    perlin_proba=0.50
)

print(f"📊 Taille du Dataset d'entraînement : {len(train_dataset)} échantillons.")

In [ ]:
# 5. Contrôle Visuel sur 5 Échantillons Augmentés
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i in range(min(5, len(train_dataset))):
    img_tensor, target, patient_id = train_dataset[i]
    # Dénormalisation pour affichage
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    
    axes[i].imshow(img_np)
    axes[i].set_title(f"Target: {target} | Patient: {patient_id}")
    axes[i].axis('off')

plt.tight_layout()
plt.savefig("../outputs/figures/dataset_samples_preview.png", dpi=150)
plt.show()
print("🎨 Aperçu sauvegardé dans ../outputs/figures/dataset_samples_preview.png")